# Phase 3 — RAG layer: indexing + retrieval

This notebook does NOT need a GPU — embedding with `all-MiniLM-L6-v2` and querying a local Qdrant collection both run fine on CPU. Runtime > Change runtime type > CPU is fine here, unlike `02_finetune.ipynb`.

Builds a searchable vector index over the full report corpus (not a train/val/test split — this simulates a real historical case archive), then tests that retrieval actually surfaces sensible similar reports before any LLM generation gets wired on top.

## 1. Mount Drive and pull the repo

Same Drive-persisted setup as the other notebooks.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os

from google.colab import userdata

GH_TOKEN = userdata.get("GH_TOKEN")
REPO_DIR = "/content/drive/MyDrive/Clinical_Report_Assistant"
REPO_URL = f"https://{GH_TOKEN}@github.com/ozgurberat/clinical-report-assistant.git"

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print("Repo already exists in Drive — pulling latest instead of re-cloning.")
    %cd {REPO_DIR}
    !git checkout -- notebooks/*.ipynb
    !git pull
else:
    !git clone {REPO_URL} "{REPO_DIR}"
    %cd {REPO_DIR}

## 2. Install the RAG dependencies

`sentence-transformers` (the embedding model) and `qdrant-client` (the local vector store) — neither needs a GPU.

In [ ]:
!pip install -q sentence-transformers qdrant-client pyyaml

## 3. Build the index

Embeds every report in `data/processed/reports.jsonl` and writes a local Qdrant collection to `data/processed/qdrant_index/`. Takes a few minutes on CPU for ~3,400 reports — this only needs to run once (or again if the underlying corpus changes); the index persists in Drive afterward.

In [ ]:
!python -m src.rag.build_index --processed data/processed

## 4. Test retrieval — does this actually surface similar reports?

Before building anything on top of this, read the actual matches by eye. A good sign: the top hits should share real clinical similarity with the query (similar findings/diagnosis), not just superficial word overlap — that's the whole point of embedding-based search over keyword search.

In [ ]:
!python -m src.rag.retrieve --query "mild cardiomegaly, clear lungs" --top-k 5

In [ ]:
!python -m src.rag.retrieve --query "pneumothorax with chest tube in place" --top-k 5

Retrieval looks solid — top matches are genuinely clinically similar, not just coincidental word overlap. Time to wire an actual answer on top of it.

## 5. Switch to a GPU runtime

Everything above (embedding, indexing, retrieval) ran fine on CPU. This next part loads Qwen3-4B to actually generate an answer, so it needs a GPU again — same as `02_finetune.ipynb`. Runtime > Change runtime type > GPU (T4 or A100), then continue from here.

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU detected — check Runtime > Change runtime type"
print(torch.cuda.get_device_name(0))

## 6. Install the generation dependencies

Same quantization stack as fine-tuning, minus `peft` — this step deliberately uses the plain base model with no adapter attached (see `src/rag/qa.py`'s docstring for why).

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes

## 7. Ask a question, end to end

Retrieves the most similar past cases, builds a prompt embedding them as context, and generates an answer with the base Qwen3-4B model — thinking mode left ON this time (unlike the fine-tuned generation in `02_finetune.ipynb`), since synthesizing across several retrieved documents is exactly the kind of multi-step task it's meant to help with. `--show-reasoning` prints the model's actual reasoning trace so you can judge whether it's really using the retrieved evidence or just producing something plausible-sounding.

In [ ]:
!python -m src.rag.qa --show-reasoning --question "This new patient has mild cardiomegaly with clear lungs — what have we typically found in similar past cases, and what usually got recommended as follow-up?"

## Next

Read the reasoning trace and the final answer together. Two things worth checking specifically: does the answer actually reference the retrieved report IDs / findings rather than generic textbook knowledge, and does it stay honest when the retrieved evidence is thin rather than confidently overstating? If this looks good, Phase 3 is functionally done and we can move to Phase 4 (FastAPI serving + Docker). If the answer ignores the retrieved context or the reasoning looks off, bring back the actual output and we'll dig into the prompt or retrieval quality before moving on.